[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C66_Agentic_Evaluation_Course/05_cost_harness/05_cost_and_harness.ipynb)

# 05 · 成本感知评测与 harness 可复现性（成本模型 / 帕累托前沿 / best-of-n / scaffold 2×2 / 指纹 / 评测卡）

目标：把「谁更强」这个没有定义的问题，改写成「给定预算谁更强」，并把 harness 钉死到别人能复现。

本 notebook 你会亲手实现：
1. **成本模型与归一化** —— 四种成本，以及为什么必须报「每次成功的成本」
2. **成本-成功率帕累托前沿** —— 找出被支配的配置，按预算线选配置
3. **best-of-n 的边际收益与陷阱** —— 判分器假阳率如何在大 n 时系统性选出 hack 解
4. **scaffold 2×2 实验** —— 交互项如何翻转「哪个模型更强」的结论
5. **运行指纹与漂移检测** —— 十项清单压成一个哈希，让「不可比较」变成机器可判定
6. **评测卡生成器** —— 从原始结果直接产出一张完整的 eval card

> 心智模型：**agent 的能力不是一个标量，是一条成本-成功率曲线。
> 两条曲线可以相交——所以「哪个模型最强」经常没有答案，而「我们的预算下选哪个」永远有答案。**

## 1 · 成本模型与归一化

In [ ]:
import math, json, hashlib, itertools
from collections import defaultdict
import numpy as np

PRICE = {'in_per_mtok': 3.0, 'out_per_mtok': 15.0}     # 美元 / 百万 token（示例价）

def task_cost(tokens_in, tokens_out, tool_calls=0, container_sec=0.0,
              price=PRICE, tool_unit=0.0002, container_per_sec=0.00012):
    return (tokens_in / 1e6 * price['in_per_mtok']
            + tokens_out / 1e6 * price['out_per_mtok']
            + tool_calls * tool_unit
            + container_sec * container_per_sec)

def run_costs(rows):
    """rows: [{'score':0/1,'tokens_in':..,'tokens_out':..,'tool_calls':..,'container_sec':..}]
    返回三种口径：每任务成本 / 每次成功成本 / 每美元买到的成功数。"""
    total = sum(task_cost(r['tokens_in'], r['tokens_out'], r['tool_calls'], r['container_sec'])
                for r in rows)
    n_succ = sum(r['score'] for r in rows)
    return {'per_task': total / len(rows),
            'per_success': total / n_succ if n_succ else float('inf'),
            'success_per_dollar': n_succ / total if total else 0.0}

rng = np.random.default_rng(0)

def make_rows(n, p_succ, mean_tok_succ, mean_tok_fail, rng):
    rows = []
    for _ in range(n):
        ok = rng.random() < p_succ
        base = mean_tok_succ if ok else mean_tok_fail
        ti = int(rng.normal(base, base * 0.25))
        rows.append({'score': float(ok), 'tokens_in': max(ti, 100),
                     'tokens_out': max(int(ti * 0.05), 20),
                     'tool_calls': int(max(ti / 4000, 1)), 'container_sec': ti / 900})
    return rows

# A：成功率高、失败时也会硬撑到底   B：成功率低得多、一遇到麻烦就早早放弃
A = make_rows(400, 0.52, 90000, 120000, rng)
B = make_rows(400, 0.22, 80000, 45000, rng)
for name, rows in [('A 稳健', A), ('B 早放弃', B)]:
    c = run_costs(rows)
    print(f"{name:<10} 成功率 {np.mean([r['score'] for r in rows]):.1%} | "
          f"每任务 ${c['per_task']:.3f} | 每次成功 ${c['per_success']:.3f} | "
          f"每美元成功 {c['success_per_dollar']:.2f}")

cA, cB = run_costs(A), run_costs(B)
assert cB['per_task'] < cA['per_task'], 'B 的每任务成本更低（失败得快，一半的钱都没花出去）'
assert cB['per_success'] > cA['per_success'], '但 B 的每次成功成本更高（成功率太低，摊不动）'
print('\n✅ 两个口径给出相反的结论。「每任务成本」奖励早放弃——这是模块 03 Goodhart 的成本版本。')
print('   报告里唯一可比的口径是「每次成功的成本」。')

## 2 · 成本-成功率帕累托前沿

In [ ]:
def pareto_front(points):
    """points: [(name, cost, success)]。返回不被任何其他点支配的点
    （支配 = 成本更低 且 成功率更高）。"""
    front = []
    for name, c, s in points:
        dominated = any((c2 <= c and s2 >= s) and (c2 < c or s2 > s)
                        for n2, c2, s2 in points if n2 != name)
        if not dominated:
            front.append((name, c, s))
    return sorted(front, key=lambda t: t[1])

CONFIGS = [
    ('A · haiku + minimal',      0.05, 0.22),
    ('B · haiku + reflect',      0.20, 0.38),
    ('C · sonnet + minimal',     0.42, 0.47),
    ('D · sonnet + reflect',     1.05, 0.58),
    ('E · sonnet + 4 subagents', 4.10, 0.55),     # 更贵却更差 → 被 D 支配
    ('F · haiku + 8 retries',    0.60, 0.35),     # 被 C 支配
    ('G · opus + reflect',      18.00, 0.64),
]
front = pareto_front(CONFIGS)
print('帕累托前沿（按成本升序）:')
for n, c, s in front:
    print(f'  {n:<28} ${c:>6.2f}  {s:.0%}')
dominated = [n for n, c, s in CONFIGS if n not in [f[0] for f in front]]
print(f'\n被支配（任何预算下都不该选）: {dominated}')
assert 'E · sonnet + 4 subagents' in dominated
assert 'F · haiku + 8 retries' in dominated
print('✅ E 和 F 在任何预算下都不该被选——这类点在真实评测里非常常见，')
print('   通常意味着某个 scaffold 没调好，而不是模型不行。')

In [ ]:
def pick_under_budget(points, budget):
    ok = [(n, c, s) for n, c, s in points if c <= budget]
    return max(ok, key=lambda t: t[2]) if ok else None

def cheapest_for_target(points, target):
    ok = [(n, c, s) for n, c, s in points if s >= target]
    return min(ok, key=lambda t: t[1]) if ok else None

print('按预算选:')
for b in [0.10, 0.50, 2.00, 20.0]:
    print(f'  预算 ${b:>6.2f} → {pick_under_budget(CONFIGS, b)}')
print('\n按目标选:')
for t in [0.35, 0.50, 0.60]:
    print(f'  目标 {t:.0%} → {cheapest_for_target(CONFIGS, t)}')

# 边际成本：沿前沿往上走，每多 1 个百分点要多花多少钱
print('\n沿前沿的边际成本（每 +1 个百分点的成功率要多花多少）:')
for (n1, c1, s1), (n2, c2, s2) in zip(front, front[1:]):
    print(f'  {n1[:12]:<12} → {n2[:12]:<12}  ${(c2-c1)/((s2-s1)*100):>7.3f} / 百分点')

assert pick_under_budget(CONFIGS, 0.10)[0].startswith('A')
assert pick_under_budget(CONFIGS, 20.0)[0].startswith('G')
print('\n✅ 注意最后一行的边际成本——从 D 到 G 每提高一个百分点要花几美元。')
print('   这个数字才是「值不值得换更强模型」这个决策的真正输入。')

## 3 · best-of-n：边际收益递减，以及判分器假阳率的放大效应

In [ ]:
def pass_at_n(p, n):
    return 1 - (1 - p) ** n

P_SINGLE = 0.35
print(f"{'n':>4}{'pass@n':>10}{'相对成本':>10}{'每+1点的边际成本':>20}")
prev_s, prev_c = P_SINGLE, 1.0
for n in [1, 2, 4, 8, 16, 32]:
    s, c = pass_at_n(P_SINGLE, n), float(n)
    marg = (c - prev_c) / ((s - prev_s) * 100) if n > 1 and s > prev_s else float('nan')
    print(f'{n:>4}{s:>10.1%}{c:>10.1f}x{marg:>19.3f}')
    prev_s, prev_c = s, c

assert pass_at_n(P_SINGLE, 2) - pass_at_n(P_SINGLE, 1) > \
       pass_at_n(P_SINGLE, 32) - pass_at_n(P_SINGLE, 16)
print('\n✅ 从 1 到 2 涨 12 个点，从 16 到 32 只涨不到 1 个点，而成本翻倍。')

In [ ]:
def bon_with_imperfect_verifier(p_true, n, alpha, rng, trials=5000):
    """验证器有假阳率 alpha（把失败解判成成功）。best-of-n 会挑第一个被判成功的候选。
    返回 (被判成功的比例, 在被判成功的样本中真正正确的比例)。"""
    picked, correct = 0, 0
    for _ in range(trials):
        real = rng.random(n) < p_true                 # 每个候选是否真的正确
        judged = np.where(real, True, rng.random(n) < alpha)   # 验证器判定（真解一律判对）
        if judged.any():
            picked += 1
            idx = int(np.argmax(judged))              # 取第一个被判成功的
            correct += bool(real[idx])
    return picked / trials, (correct / picked if picked else float('nan'))

rng = np.random.default_rng(3)
print(f"{'n':>4}{'验证器说成功':>14}{'其中真正正确':>14}{'真实成功率':>12}")
for n in [1, 2, 4, 8, 16, 32]:
    sel, prec = bon_with_imperfect_verifier(0.35, n, alpha=0.05, rng=rng)
    print(f'{n:>4}{sel:>14.1%}{prec:>14.1%}{sel*prec:>12.1%}')

sel1, prec1 = bon_with_imperfect_verifier(0.35, 1, 0.05, rng)
sel32, prec32 = bon_with_imperfect_verifier(0.35, 32, 0.05, rng)
assert abs(prec32 - prec1) < 0.05, '被选中解的精确率恒为 p/q，与 n 无关'
assert sel32 > sel1
gap1, gap32 = sel1 * (1 - prec1), sel32 * (1 - prec32)
print(f'\n✅ 精确率恒为 p/q ≈ {prec1:.0%}，与 n 无关；')
print(f'   但验证器报出的成功率从 {sel1:.0%} 一路涨到 {sel32:.0%}（看起来很棒）。')
print(f'   两者之差就是被系统性引入的假阳解：n=1 时占 {gap1:.1%}，n=32 时占 {gap32:.1%}。')
assert gap32 > gap1
print('   **n 越大，你交付出去的"成功"里越多是判分器的假阳解**——')
print('   这就是 reward hacking 在评测侧的同构现象（C67-05 会从奖励模型角度重讲）。')

## 4 · scaffold 2×2 实验：交互项如何翻转结论

In [ ]:
# 两个模型 × 两个 scaffold。真值由一个「模型能力 + scaffold 加成 + 交互项」的模型生成。
TRUTH = {
    ('model_A', 'minimal'): 0.30,
    ('model_A', 'reflect'): 0.52,       # A 从反思循环里获益极大
    ('model_B', 'minimal'): 0.38,
    ('model_B', 'reflect'): 0.44,       # B 获益有限（它本来就会自我检查）
}

def run_cell(p, N=600, seed=0):
    rng = np.random.default_rng(seed)
    return (rng.random(N) < p).astype(float)

cells = {key: run_cell(v, seed=100 + i) for i, (key, v) in enumerate(sorted(TRUTH.items()))}
print(f"{'':<10}{'minimal':>10}{'reflect':>10}")
for m in ['model_A', 'model_B']:
    print(f"{m:<10}{cells[(m,'minimal')].mean():>10.1%}{cells[(m,'reflect')].mean():>10.1%}")

winner_minimal = 'model_A' if cells[('model_A', 'minimal')].mean() > cells[('model_B', 'minimal')].mean() else 'model_B'
winner_reflect = 'model_A' if cells[('model_A', 'reflect')].mean() > cells[('model_B', 'reflect')].mean() else 'model_B'
print(f'\nminimal scaffold 下的赢家: {winner_minimal}')
print(f'reflect scaffold 下的赢家: {winner_reflect}')
assert winner_minimal != winner_reflect
print('\n✅ 换一套 scaffold，「哪个模型更强」的结论直接翻转。')
print('   只报一个 scaffold 下的分数，等于在报告一个由你自己的实现决定的结论。')

In [ ]:
# 效应分解：主效应 vs 交互项
def effects(cells):
    a_min, a_ref = cells[('model_A', 'minimal')].mean(), cells[('model_A', 'reflect')].mean()
    b_min, b_ref = cells[('model_B', 'minimal')].mean(), cells[('model_B', 'reflect')].mean()
    model_effect = ((a_min + a_ref) - (b_min + b_ref)) / 2
    scaffold_effect = ((a_ref + b_ref) - (a_min + b_min)) / 2
    interaction = (a_ref - a_min) - (b_ref - b_min)
    return model_effect, scaffold_effect, interaction

me, se, ix = effects(cells)
print(f'模型主效应 (A - B)      {me:+.1%}')
print(f'scaffold 主效应 (ref-min) {se:+.1%}')
print(f'交互项                  {ix:+.1%}')
assert abs(se) > abs(me), 'scaffold 的主效应大于模型的主效应'
assert abs(ix) > abs(me), '交互项也大于模型主效应'
print('\n✅ scaffold 的效应和交互项都大于模型主效应——')
print('   这就是「agent 基准测的是「模型+你写的那个程序」的组合」这句话的定量形式。')
print('   报告规范：至少给出一个最简 baseline scaffold 下的分数作为参照。')

## 5 · 运行指纹与漂移检测

In [ ]:
REPRO_FIELDS = [
    'image_digest', 'dataset_hash', 'scaffold_commit', 'model_id',
    'temperature', 'max_steps', 'max_tokens', 'timeout_s', 'retries',
    'concurrency', 'network_policy', 'scorer_commit', 'n_tasks', 'k_attempts',
    'aggregation', 'ci_method',
]

def fingerprint(cfg):
    missing = [f for f in REPRO_FIELDS if f not in cfg]
    if missing:
        raise ValueError(f'配置缺失字段，无法生成指纹: {missing}')
    payload = json.dumps({f: cfg[f] for f in REPRO_FIELDS}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:8]

BASE = {'image_digest': 'sha256:9a1c', 'dataset_hash': 'sha256:7d2e',
        'scaffold_commit': 'a3f19c2', 'model_id': 'claude-sonnet-5@2026-05',
        'temperature': 0.0, 'max_steps': 40, 'max_tokens': 200000,
        'timeout_s': 300, 'retries': 2, 'concurrency': 8,
        'network_policy': 'offline', 'scorer_commit': 'b71d0e4',
        'n_tasks': 420, 'k_attempts': 5, 'aggregation': 'micro',
        'ci_method': 'cluster_bootstrap'}

fp_base = fingerprint(BASE)
print('baseline 指纹:', fp_base)
for change in [{'concurrency': 16}, {'max_steps': 60}, {'model_id': 'claude-sonnet-5@2026-08'},
               {'aggregation': 'macro'}]:
    cfg = dict(BASE, **change)
    print(f'  改 {list(change)[0]:<16} → {fingerprint(cfg)}  '
          f'{"⚠️ 不可与 baseline 直接比较" if fingerprint(cfg) != fp_base else ""}')

assert fingerprint(BASE) == fingerprint(dict(BASE))
try:
    fingerprint({k: v for k, v in BASE.items() if k != 'concurrency'})
    raise AssertionError('缺字段时应当报错')
except ValueError as e:
    print(f'\n缺字段时正确报错: {str(e)[:46]}…')
print('✅ 指纹机制让「不可比较」变成机器可判定的——')
print('   C68 的 CI 门禁里这直接是一行 assert，而不是让人肉眼核对配置。')

In [ ]:
def drift_alarm(runs, tol=0.05):
    """同指纹的多次运行，分数差异超过 tol 就是告警——说明还有没被记录的变量在动。"""
    by_fp = defaultdict(list)
    for r in runs:
        by_fp[r['fingerprint']].append(r['score'])
    alarms = []
    for fp, scores in by_fp.items():
        if len(scores) > 1 and (max(scores) - min(scores)) > tol:
            alarms.append((fp, min(scores), max(scores), max(scores) - min(scores)))
    return alarms

RUNS = [
    {'fingerprint': fp_base, 'score': 0.417},
    {'fingerprint': fp_base, 'score': 0.424},
    {'fingerprint': fp_base, 'score': 0.489},        # ← 同配置却高出 7 个点
    {'fingerprint': 'deadbeef', 'score': 0.512},
]
al = drift_alarm(RUNS)
for fp, lo, hi, d in al:
    print(f'⚠️ 指纹 {fp} 的多次运行分数从 {lo:.1%} 到 {hi:.1%}（差 {d:.1%}）——存在未记录的变量')
assert len(al) == 1
print('\n✅ 这类告警的常见根因：外部 API 行为变了、容器所在机器负载不同导致超时数变了、')
print('   或者某个「以为是常量」的东西其实读了环境变量。指纹机制让它们暴露出来。')

## 6 · 评测卡生成器

In [ ]:
def eval_card(name, cfg, X, rows, alpha_beta, baseline_scaffold_score=None):
    """从原始结果直接产出一张 eval card（字典形式，print 出来就是报告）。"""
    X = np.asarray(X, dtype=float)
    N, k = X.shape
    task_means = X.mean(axis=1)
    vb = float(np.var(task_means, ddof=1))
    vw = float(np.mean(task_means * (1 - task_means)))
    rho = vb / (vb + vw) if (vb + vw) > 0 else 0.0
    n_eff = N * k / (1 + (k - 1) * rho)
    rng = np.random.default_rng(0)
    boots = [X[rng.integers(0, N, N)].mean() for _ in range(1500)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    a, b = alpha_beta
    p_obs = float(X.mean())
    corrected = (p_obs - a) / (1 - b - a)
    costs = run_costs(rows)
    return {
        'name': name,
        'fingerprint': fingerprint(cfg),
        'N_tasks': N, 'k_attempts': k,
        'pass1_micro': round(p_obs, 4),
        'ci95': (round(float(lo), 4), round(float(hi), 4)),
        'scorer_alpha_beta': (a, b),
        'pass1_corrected': round(corrected, 4),
        'n_eff': round(n_eff, 1), 'rho': round(rho, 3),
        'usd_per_task': round(costs['per_task'], 4),
        'usd_per_success': round(costs['per_success'], 4),
        'baseline_scaffold_pass1': baseline_scaffold_score,
    }

rng = np.random.default_rng(8)
Xdemo = (rng.random((420, 5)) < rng.beta(1.2, 1.6, size=(420, 1))).astype(float)
card = eval_card('agent-billing-v3', BASE, Xdemo, A, alpha_beta=(0.031, 0.022),
                 baseline_scaffold_score=0.301)
print('# ═════════ EVAL CARD ═════════')
for kk, vv in card.items():
    print(f'{kk:<26} {vv}')

assert card['pass1_corrected'] < card['pass1_micro'], '假阳率 > 假阴率时，校正后应下降'
assert card['n_eff'] <= 420 * 5
assert card['usd_per_success'] >= card['usd_per_task']
print('\n✅ 这张卡里最容易被砍掉、也最不能砍的三行：scorer_alpha_beta / n_eff / fingerprint。')
print('   它们的共同点是「限制你能从数字里得出的结论」——所以在追求好看的压力下最先被删。')

## ✏️ 练习 1：等预算下的公平对比

实现 `equalize_budget(configs, budget)`：`configs` 是 `[(name, cost_per_run, p_single)]`，
在总预算 `budget` 下，每个配置能跑 `n = floor(budget / cost_per_run)` 次 best-of-n，
返回 `[(name, n, pass_at_n)]` 按 pass@n 降序。这才是「同样的钱谁更强」。

In [ ]:
def equalize_budget(configs, budget):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
CFG = [('cheap_weak', 0.05, 0.22), ('mid', 0.42, 0.47), ('expensive_strong', 4.00, 0.62)]
res = equalize_budget(CFG, budget=4.00)
print('预算 $4.00 时:')
for n_, k_, s_ in res:
    print(f'  {n_:<20} 可跑 {k_:>3} 次 → pass@n {s_:.1%}')
assert res == sorted(res, key=lambda t: -t[2]), '返回必须按 pass@n 降序'
assert res[0][0] == 'cheap_weak', '小预算下，便宜模型靠多跑几次反超'
res2 = equalize_budget(CFG, budget=0.50)
print(f'\n预算 $0.50 时的赢家: {res2[0][0]} (pass@n {res2[0][2]:.1%})')
assert dict((r[0], r[1]) for r in res)['expensive_strong'] == 1
print('✅ 练习 1 通过：等预算下，便宜模型的 best-of-n 可以反超贵模型的单次运行——')
print('   前提是你有验证器。没有验证器的话这个反超是幻觉（第 3 节）。')

## ✏️ 练习 2：帕累托前沿的支配关系检查

实现 `dominates(p, q)`：`p, q` 均为 `(name, cost, success)`，
当 `p` 的成本 ≤ `q` 且成功率 ≥ `q`，且至少一项严格更优时返回 True。
再实现 `dominated_by(point, points)` 返回所有支配它的点的名字列表。

In [ ]:
def dominates(p, q):
    # TODO
    raise NotImplementedError

def dominated_by(point, points):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
p1 = ('X', 1.0, 0.50); p2 = ('Y', 2.0, 0.40); p3 = ('Z', 1.0, 0.50)
assert dominates(p1, p2) is True
assert dominates(p2, p1) is False
assert dominates(p1, p3) is False          # 完全相同 → 互不支配
e = ('E · sonnet + 4 subagents', 4.10, 0.55)
doms = dominated_by(e, CONFIGS)
print(f'支配 E 的配置: {doms}')
assert 'D · sonnet + reflect' in doms
assert dominated_by(('G · opus + reflect', 18.00, 0.64), CONFIGS) == []
print('✅ 练习 2 通过：G 最贵但没人支配它（它也最准）——')
print('   前沿上的点都不被支配，选哪个取决于预算线，不取决于「谁分数最高」。')

## ✏️ 练习 3：判分器假阳率下 best-of-n 的真实收益

实现 `true_bon_success(p_true, n, alpha)`：解析地算出
「验证器判为成功」的概率与「被选中的解真正正确」的概率之积，即真实成功率。

提示：验证器判某个候选成功的概率 $q = p + (1-p)\alpha$；
$n$ 个候选至少一个被判成功的概率 $1-(1-q)^n$；
在被判成功的候选中，真正正确的比例是 $p / q$（对每个候选独立成立，
因此第一个被判成功的候选正确的条件概率也是 $p/q$）。

In [ ]:
def true_bon_success(p_true, n, alpha):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(true_bon_success(0.35, 1, 0.0) - 0.35) < 1e-12
assert abs(true_bon_success(0.35, 1, 0.05) - 0.35) < 1e-12    # n=1 时假阳不改变真实成功率
print(f"{'n':>4}{'α=0':>10}{'α=0.05':>10}{'α=0.20':>10}")
for n in [1, 2, 4, 8, 16, 32]:
    print(f'{n:>4}', end='')
    for a_ in [0.0, 0.05, 0.20]:
        print(f'{true_bon_success(0.35, n, a_):>10.1%}', end='')
    print()
assert true_bon_success(0.35, 32, 0.20) < true_bon_success(0.35, 32, 0.0)
gap = true_bon_success(0.35, 32, 0.0) - true_bon_success(0.35, 32, 0.20)
assert gap > 0.2
print(f'\nn=32 时，α 从 0 涨到 0.20 让真实成功率掉了 {gap:.0%}')
print('✅ 练习 3 通过：best-of-n 的收益完全建立在验证器质量上——')
print('   验证器越差，n 越大，你挑出来的越可能是假阳解。')

## ✏️ 练习 4：上线决策的期望效用

实现 `expected_utility(p_success, v_success, c_fail, c_run)`：
返回 $p \cdot V - (1-p)\cdot C_{fail} - C_{run}$。
再实现 `min_success_for_launch(v_success, c_fail, c_run, baseline_utility=0.0)`：
反解上线所需的最低成功率，结果夹到 $[0,1]$。

In [ ]:
def expected_utility(p_success, v_success, c_fail, c_run):
    # TODO
    raise NotImplementedError

def min_success_for_launch(v_success, c_fail, c_run, baseline_utility=0.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert abs(expected_utility(1.0, 10, 50, 1) - 9.0) < 1e-12
assert abs(expected_utility(0.0, 10, 50, 1) + 51.0) < 1e-12
p_need = min_success_for_launch(v_success=10, c_fail=50, c_run=1)
assert abs(expected_utility(p_need, 10, 50, 1)) < 1e-9
print(f"{'场景':<28}{'C_fail':>9}{'上线门槛':>12}")
for label, cf in [('草稿建议（人来定稿）', 2), ('自动执行、可撤销', 50), ('自动执行、不可撤销', 500)]:
    print(f'{label:<28}{cf:>9}{min_success_for_launch(10, cf, 1):>12.1%}')
assert min_success_for_launch(10, 500, 1) > min_success_for_launch(10, 2, 1)
assert min_success_for_launch(10, 500, 20) == 1.0   # 运行成本 > 一次成功的价值 → 任何成功率都不划算
print(f'\n运行成本(20) 超过一次成功的价值(10) 时的门槛: '
      f'{min_success_for_launch(10, 500, 20):.0%}（夹到上限）')
print('✅ 练习 4 通过：门槛随失败代价急剧上升——不可撤销场景要到 98%，')
print('   而当运行成本本身就超过一次成功的价值时，门槛被夹到 100%，')
print('   意思是「靠提高成功率解决不了」：要么降成本，要么加人工确认点。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def equalize_budget(configs, budget):
    out = []
    for name, cost, p in configs:
        n = max(int(budget // cost), 1)
        out.append((name, n, pass_at_n(p, n)))
    return sorted(out, key=lambda t: -t[2])

In [ ]:
# 练习 2 参考答案
def dominates(p, q):
    return (p[1] <= q[1] and p[2] >= q[2]) and (p[1] < q[1] or p[2] > q[2])

def dominated_by(point, points):
    return [n for n, c, s in points if n != point[0] and dominates((n, c, s), point)]

In [ ]:
# 练习 3 参考答案
def true_bon_success(p_true, n, alpha):
    q = p_true + (1 - p_true) * alpha        # 单个候选被判为成功的概率
    if q <= 0:
        return 0.0
    p_any_judged = 1 - (1 - q) ** n
    precision = p_true / q                    # 被判成功的候选里真正正确的比例
    return p_any_judged * precision

In [ ]:
# 练习 4 参考答案
def expected_utility(p_success, v_success, c_fail, c_run):
    return p_success * v_success - (1 - p_success) * c_fail - c_run

def min_success_for_launch(v_success, c_fail, c_run, baseline_utility=0.0):
    # p*V - (1-p)*C_fail - C_run = baseline  →  p*(V + C_fail) = baseline + C_run + C_fail
    denom = v_success + c_fail
    if denom <= 0:
        return 1.0
    p = (baseline_utility + c_run + c_fail) / denom
    return float(min(1.0, max(0.0, p)))

---
## 🧪 真实工程胶囊：可复现评测的落地骨架

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 钉死容器：永远用 digest，永远不用 tag
# ══════════════════════════════════════════════════════════════════
# ✗ docker run swebench/eval:latest
# ✓ docker run swebench/eval@sha256:9a1c7f...   ← digest 不可变
# 取 digest:  docker inspect --format='{{index .RepoDigests 0}}' <image>
# 在评测配置里存 digest，并在每条结果记录里带上它。

# ══════════════════════════════════════════════════════════════════
# B. 并发度是 harness 的一部分（最常被忽略的一条）
# ══════════════════════════════════════════════════════════════════
# 高并发 → 容器争抢 CPU → 测试跑得慢 → 更多任务撞上 timeout → 分数下降。
# 复现清单里必须写 concurrency，并且做一次敏感性检查：
#   for c in [1, 4, 8, 16]:
#       run_eval(..., concurrency=c)      # 分数随 c 变化 = 你的 timeout 设得太紧
# 分数对并发不敏感，才说明 timeout 有足够余量。

# ══════════════════════════════════════════════════════════════════
# C. 网络：录制回放
# ══════════════════════════════════════════════════════════════════
# pip install vcrpy
import vcr
my_vcr = vcr.VCR(record_mode="once", match_on=["method", "scheme", "host", "path", "query"],
                 filter_headers=["authorization", "x-api-key"])   # ← 千万别把密钥录进去
with my_vcr.use_cassette("cassettes/task_017.yaml"):
    run_agent(task_017)
# 双轨制：录制回放跑回归门禁（可复现），每周一次真实联网小样本校验录制没失真。

# ══════════════════════════════════════════════════════════════════
# D. 每条结果记录的最小字段（够算出本课全部指标）
# ══════════════════════════════════════════════════════════════════
RESULT_ROW = {
  "run_id": "...", "fingerprint": "f2a91b7c",
  "task_id": "...", "attempt": 0,
  "score": 1, "scorer_version": "b71d0e4",
  "steps": 14, "tokens_in": 51230, "tokens_out": 2210,
  "tool_calls": 12, "container_sec": 61.2, "wall_ms": 41200,
  "failure_class": None,          # 模块 03 的七分类
  "terminated_by": None,          # loop_detector | budget | timeout | done
}

# ══════════════════════════════════════════════════════════════════
# E. 报告前的自检清单（六个 yes 才能发出去）
# ══════════════════════════════════════════════════════════════════
# [ ] 判分器的 alpha/beta 量过了吗？（模块 02）
# [ ] N 和 k 分开写了吗？CI 用的是聚类自举吗？（模块 04）
# [ ] MDE 写了吗？（模块 04）
# [ ] 成本报的是 per_success 而不是 per_task 吗？（本模块）
# [ ] baseline scaffold 的分数给了吗？（本模块第 4 节）
# [ ] fingerprint 写进报告了吗？（本模块第 5 节）
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 成本是一等公民 | 不带预算的「谁更强」没有定义 | 任何模型对比 |
| 归一化口径 | 报 `$/success` 而不是 `$/task`——后者奖励早放弃 | 报告规范 |
| 帕累托前沿 | 被支配的配置在任何预算下都不该选 | 模型选型 |
| best-of-n | 收益急剧递减，且对判分器假阳率极度敏感 | 决定要不要多采样 |
| scaffold 2×2 | scaffold 效应与交互项常大于模型主效应 | 实验设计 |
| 十项复现清单与指纹 | 让「不可比较」变成机器可判定的 | CI 门禁（C68） |
| 期望效用 | 失败代价高时，加人工确认点比提分更划算 | 上线决策 |

**全课到此闭环**：01 选/建任务集 → 02 定义并验证判分器 → 03 从轨迹挖归因 →
04 给出可信的不确定度 → 05 钉死成本与可复现性并翻译成决策。

**下一门课**：判分器如果本身是一个 LLM 呢？「先验证测量仪器」这套逻辑要怎么执行？——**C67 · LLM-as-a-Judge 与评分模型**。